In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\tools\jdk-17.0.18+8"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ["PATH"]
print(os.environ.get("JAVA_HOME"))

C:\tools\jdk-17.0.18+8


In [9]:
import os

# Windows Hadoop folder with winutils.exe
os.environ['HADOOP_HOME'] = r"C:\hadoop"
os.environ['PATH'] += os.pathsep + r"C:\hadoop\bin"

# Check if Python sees it
print("HADOOP_HOME:", os.environ['HADOOP_HOME'])
print("PATH includes Hadoop bin:", r"C:\hadoop\bin" in os.environ['PATH'])

HADOOP_HOME: C:\hadoop
PATH includes Hadoop bin: True


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .getOrCreate()

spark.sparkContext._jvm.java.lang.System.getProperty("java.version")

'17.0.18'

In [12]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [13]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [14]:
df = df.repartition(4)
df.write.mode("overwrite").parquet("yellow_tripdata_2025-11/")

In [15]:
print(pyspark.__version__)

3.5.8


In [16]:
import os

folder_path = "yellow_tripdata_2025-11/"
parquet_files = [f for f in os.listdir(folder_path) if f.endswith(".parquet")]
total_size_bytes = sum(os.path.getsize(os.path.join(folder_path, f)) for f in parquet_files)
num_files = len(parquet_files)
avg_size_mb = (total_size_bytes / num_files) / (1024 * 1024)

print(f"Average size per Parquet file: {avg_size_mb:.2f} MB")

Average size per Parquet file: 24.42 MB


In [17]:
from pyspark.sql.functions import col, to_date

nov15_trips = df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15")
num_trips = nov15_trips.count()

print(f"Number of trips on 2025-11-15: {num_trips}")

Number of trips on 2025-11-15: 162604


In [18]:
from pyspark.sql.functions import col, unix_timestamp, max as spark_max

# Calculate trip duration in seconds, then convert to hours
df_with_duration = df.withColumn(
    "trip_duration_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

# Find the maximum duration
max_duration = df_with_duration.select(spark_max("trip_duration_hours")).collect()[0][0]

print(f"Longest trip duration: {max_duration:.2f} hours")

Longest trip duration: 90.65 hours


In [19]:
zone_df = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")
zone_df.createOrReplaceTempView("zones")

In [23]:
spark.sql("SELECT * FROM zones WHERE Borough = 'Manhattan'").show(5)

+----------+---------+-----------------+------------+
|LocationID|  Borough|             Zone|service_zone|
+----------+---------+-----------------+------------+
|         4|Manhattan|    Alphabet City| Yellow Zone|
|        12|Manhattan|     Battery Park| Yellow Zone|
|        13|Manhattan|Battery Park City| Yellow Zone|
|        24|Manhattan|     Bloomingdale| Yellow Zone|
|        41|Manhattan|   Central Harlem|   Boro Zone|
+----------+---------+-----------------+------------+
only showing top 5 rows



In [21]:
from pyspark.sql import functions as F

df.join(zone_df, df.PULocationID == zone_df.LocationID, "left") \
  .groupBy("Zone") \
  .agg(F.count("*").alias("trip_count")) \
  .orderBy("trip_count") \
  .show(1)

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+
only showing top 1 row

